In [32]:
# (필요 시) 최초 1회 설치
# !pip install konlpy scikit-learn pandas numpy

import re
import numpy as np
import pandas as pd

from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [33]:
df1 = pd.read_json("data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")

In [34]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50336 entries, 0 to 50335
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        50336 non-null  object
 1   카테고리       50336 non-null  object
 2   대화셋일련번호    50336 non-null  object
 3   화자         50336 non-null  object
 4   문장번호       50336 non-null  int64 
 5   고객의도       50336 non-null  object
 6   상담사의도      50336 non-null  object
 7   QA         50336 non-null  object
 8   고객질문(요청)   50336 non-null  object
 9   상담사질문(요청)  50336 non-null  object
 10  고객답변       50336 non-null  object
 11  상담사답변      50336 non-null  object
 12  개체명        50336 non-null  object
 13  용어사전       50336 non-null  object
 14  지식베이스      50336 non-null  object
dtypes: int64(1), object(14)
memory usage: 5.8+ MB


In [83]:
df1['개체명 '].value_counts()

개체명 
                               9874
신청                              427
궁금                              419
질문                              224
문의                              218
                               ... 
부스                                1
부스, 예정                            1
연소득, 합산,이하                        1
학생, 체험, 공예품, 만들기                  1
코로나, 예방, 사람, 방문, 예상, 폐쇄, 조치       1
Name: count, Length: 19070, dtype: int64

In [35]:
test_df = df1.iloc[:, [2, 7, 8, 9, 10, 11]]

In [36]:
# 우리 데이터셋 기준 기본 컬럼명
QUESTION_COL = "고객질문(요청)"   # 질의(검색 기준)
ANSWER_COL   = "상담사답변"       # 검색 결과로 보여줄 텍스트


# 간단 정규화 함수 (URL/이메일 제거 + 공백 정리)
def normalize(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # URL 제거
    text = re.sub(r"\S+@\S+", " ", text)                # 이메일 제거
    text = re.sub(r"\s+", " ", text).strip()            # 공백 정리
    return text


In [37]:
test_df[QUESTION_COL] = test_df[QUESTION_COL].map(normalize)
test_df[ANSWER_COL]   = test_df[ANSWER_COL].map(normalize)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_6044\3029044837.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[QUESTION_COL] = test_df[QUESTION_COL].map(normalize)
C:\Users\ekfla\AppData\Local\Temp\ipykernel_6044\3029044837.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df[ANSWER_COL]   = test_df[ANSWER_COL].map(normalize)


In [38]:
flag_1 = (test_df['고객질문(요청)'] != '') & (test_df.shift(-1)['상담사답변'] != '')
flag_2 = (test_df['상담사답변'] != '') & (test_df.shift(1)['고객질문(요청)'] != '')

In [39]:
test_df.head(10)

,대화셋일련번호,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변
0,B2240,Q,지방세를 내려면 어떻게 해야됩니까?,,,
1,B2240,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
2,B2240,Q,은행 어플에서도 됩니까?,,,
3,B2240,Q,,어떤 은행을 이용하고 계십니까?,,
4,B2240,A,,,기업은행을 이용하고 있습니다.,
5,B2240,A,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.
6,B2240,Q,은행을 직접방문해도 됩니까?,,,
7,B2240,A,,,,방문납부도 가능합니다.
8,B2240,Q,은행위치 좀 알 수 있습니까?,,,
9,B2240,Q,,어느지점으로 안내해드릴까요?,,


In [40]:
q_df = test_df.loc[flag_1] 

In [41]:
q_df['상담사답변'] = test_df.loc[flag_2, '상담사답변'].to_list()

C:\Users\ekfla\AppData\Local\Temp\ipykernel_6044\2693898494.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  q_df['상담사답변'] = test_df.loc[flag_2, '상담사답변'].to_list()


In [42]:
q_df

,대화셋일련번호,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변
0,B2240,Q,지방세를 내려면 어떻게 해야됩니까?,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
6,B2240,Q,은행을 직접방문해도 됩니까?,,,방문납부도 가능합니다.
12,B2240,Q,버스로 가는 방법도 있습니까?,,,도보로 가시는게 빠를 것 같습니다.
14,B2240,Q,다른 납부방법도 있습니까?,,,위택스 사이트에서 납부하실수 있습니다.
16,B2240,Q,사이트 주소가 어떻게 됩니까?,,,입니다.
...,...,...,...,...,...,...
50326,B35809,Q,임대료는 어떻게 되나요?,,,62400원 입니다.
50328,B35809,Q,보증금도 있나요?,,,1423200원 입니다.
50330,B35809,Q,신청은 어떻게 하나요?,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다."
50332,B35809,Q,입주순위도 있나요?,,,1~3순위가 있습니다.


In [43]:
q_df = q_df[['고객질문(요청)', '상담사답변']].drop_duplicates()

In [44]:
df = q_df.copy()

In [45]:
# KOMORAN 초기화
komoran = Komoran()

# 품사 선택 규칙:
# - 명사(NNG, NNP)
# - 동사/형용사(VV, VA) : 기본형(어간) 포함
# 필요 시 불용어 목록 추가 가능
USER_STOPWORDS = set(["것", "수", "등"])  # 예시 (원하시는 불용어를 추가하세요)

def komoran_tokenize(text: str):
    """KOMORAN 형태소 분석 → (선택 품사만) 토큰 리스트 반환"""
    tokens = []
    for morph, tag in komoran.pos(text):
        if tag in ("NNG", "NNP", "VV", "VA", "SL", "MAG"):
            if morph not in USER_STOPWORDS and len(morph) >= 2:
                tokens.append(morph)
    return tokens

# 빠른 확인
print("샘플 토큰:", komoran_tokenize(df[QUESTION_COL].iloc[0])[:20])


샘플 토큰: ['지방세', '어떻']


In [46]:
# scikit-learn의 TfidfVectorizer에 "토큰 리스트"를 직접 넣기 위해
# tokenizer/preprocessor에 identity 함수를 사용하고 token_pattern은 None으로 둡니다.
def identity(x): return x

# 질문 텍스트를 형태소 기반 토큰으로 변환
question_tokens = [komoran_tokenize(t) for t in df[QUESTION_COL].tolist()]

vectorizer = TfidfVectorizer(
    tokenizer=identity,
    preprocessor=identity,
    token_pattern=None,        # tokenizer를 직접 쓰므로 패턴 비활성화
    ngram_range=(1, 2),        # 유니그램 + 바이그램
    min_df=2,                  # 최소 2문서 이상 등장
    max_df=0.95,               # 너무 흔한 토큰 제거
    max_features=50000         # 최대 어휘 수 제한
)

Xq = vectorizer.fit_transform(question_tokens)
print(f"[INFO] 질문 인덱스 구축 완료: shape={Xq.shape}, vocab_size={len(vectorizer.get_feature_names_out())}")


[INFO] 질문 인덱스 구축 완료: shape=(13174, 6125), vocab_size=6125


In [47]:
question_tokens

[['지방세', '어떻'],
 ['은행', '직접', '방문'],
 ['버스', '방법'],
 ['납부', '방법'],
 ['사이트', '주소', '어떻'],
 ['납부'],
 ['지방세', '조회'],
 ['어떻', '납부'],
 ['사이트', '주소', '어떻'],
 ['서울시', '주최', '페스티벌', '예정', '진행'],
 ['코로나', '축제', '취소', '이건', '취소'],
 ['환불'],
 ['환불', '수수료'],
 ['코로나', '수수료', '내야'],
 ['참석', '코로나', '걸리', '보상'],
 ['걸리', '보상'],
 ['확산', '강하', '취소'],
 ['전액', '환불'],
 ['청년', '저축', '계좌', '지금', '신청'],
 ['정책'],
 [],
 ['서울', '시민', '대상'],
 ['얼마', '추가', '적립'],
 ['모집인', '어떻'],
 ['신청', '기간'],
 ['주소지', '서울', '이면'],
 ['정도', '혜택'],
 ['3년', '동안', '계속', '만원'],
 [],
 ['조건'],
 ['작년', '결혼', '신혼부부', '조건'],
 ['대출', '상품'],
 ['은행'],
 ['버팀목', '대출', '조건'],
 ['서울', '소재', '주택'],
 ['대출', '한도', '어떻'],
 ['대출'],
 ['나머지', '대하', '그냥', '대출'],
 ['한정'],
 ['대출', '한도', '어떻'],
 ['금리', '얼마'],
 ['그렇', '혜택'],
 [],
 ['이율'],
 ['대출', '기간', '어떻'],
 ['5월', '퇴사', '연말정산'],
 ['제가', '혼자', '혼자', '야합'],
 ['제가', '준비'],
 ['확인'],
 ['나오', '방법'],
 ['국세청', '신청'],
 ['공인인증서', '필요'],
 ['공인인증서'],
 ['연말정산', '환급', '언제'],
 ['제가', '올해', '안경', '구입', '환급'],
 ['소화', '서비스',

In [48]:
def search_by_question(query: str, topk: int = 5) -> pd.DataFrame:
    """
    질의문(query)을 KOMORAN으로 토큰화하여,
    고객질문(요청) 인덱스(Xq)와 코사인 유사도 계산 → 상위 결과를 반환.
    반환 컬럼: 순위, 유사도, 고객질문(요청), 상담사답변
    """
    # 1) 질의 정규화 + 형태소 토크나이즈
    q_norm = normalize(query)
    q_tokens = [komoran_tokenize(q_norm)]  # 2차원(list of list) 형태

    # 2) TF-IDF 변환 및 유사도 계산
    q_vec = vectorizer.transform(q_tokens)
    sims = cosine_similarity(q_vec, Xq).ravel()

    # 3) 상위 Top-K 인덱스
    topk = int(min(topk, len(sims)))
    idx = np.argsort(-sims)[:topk]

    # 4) 결과 표 구성
    out = pd.DataFrame({
        "순위":   np.arange(1, topk + 1),
        "유사도": np.round(sims[idx], 4),
        "고객질문(요청)": df.iloc[idx][QUESTION_COL].values,
        "상담사답변":     df.iloc[idx][ANSWER_COL].values
    })
    return out


In [49]:
# 예시 질의어 (원하시는 질의어로 바꿔서 실험하세요)
query = "여권 재발급 방법"
search_by_question(query, topk=5)


,순위,유사도,고객질문(요청),상담사답변
0,1,0.7348,여권 발급소는 어디에 있나요?,서울시에서는 25개의 구청에서 발급이 가능합니다.
1,2,0.7348,신여권으로 재발급 받을수 있나요?,아직 발급은 안하고 있습니다
2,3,0.6584,여권 재발급 온라인으로도 가능하다고 들었는데 맞나요?,올해 7월부터 시범적으로 시행하고 있습니다.
3,4,0.6495,그럼 직접가서 여권재발급해야겠네요,아직은 그렇습니다.
4,5,0.5827,여권 재발급 비용도 알 수있나요?,"복수여권53000원,일반여권 50000원입니다."


In [50]:
querys = [
    '여권 재발급 신청 방법 알려줘',
    '여권 사진 규정이 어떻게 되나요', 
    '전입신고 인터넷으로 할 수 있습니까', 
    '주민등록등본 온라인 발급 방법', 
    '자동차 등록 이전 절차가 궁금합니다', 
    '지방세 환급금을 어디서 신청하나요'
]
for q in querys:
    # questions = search_by_question(q, topk=2)['상담사답변'].to_list()
    # answers = search_by_question(q, topk=2)['상담사답변'].to_list()
    # print(f'유사 질문 : {questions}, 답변 : {answers}')
    display(search_by_question(q, topk=3))

,순위,유사도,고객질문(요청),상담사답변
0,1,0.6936,신청방법을 알려주세요.,주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
1,2,0.6936,신청방법좀 알려주시겠어요?,네. 카드로택스를 이용하거나 세무서를 방문하여 신청하실수 있습니다.
2,3,0.6936,수강신청 방법 알려주세요.,회원 로그인 후 상단의 온라인 학습 접속 후 검색창에 강의명 확인하고 수강신청 버튼...


,순위,유사도,고객질문(요청),상담사답변
0,1,0.7479,여권 발급할떄 사진은 몇장 필요하죠?,1장 준비하시면 됩니다.
1,2,0.7443,여권 사진 전에 찍어둔 거 사용해도 되나요?,6개월 이내 찍은 사진을 사용하셔야 합니다.
2,3,0.6765,여권용 사진의 사이즈는 어떻게 되나요?,3.5*4.5cm 입니다.


,순위,유사도,고객질문(요청),상담사답변
0,1,0.9024,전입신고는 가서 해야되죠?,방문신고는 신 거주지 동주민센터에서만 가능합니다.
1,2,0.9024,전입신고는 거기가서 해야하죠?,방문신고는 신 거주지 동주민센터에서만 가능한 부분입니다.
2,3,0.6842,전입신고 비용은 얼마에요?,전입신고 수수료는 무료입니다


,순위,유사도,고객질문(요청),상담사답변
0,1,0.7634,주민등록등본을 발급 방법을 알고 싶습니다.,예.온라인과 오프라인 방법이 있습니다.
1,2,0.5916,주민등록표등본은 무료인가요?,"주민등록표등,초본은 무료발급 민원입니다."
2,3,0.5492,주민등록등본도 팩스로 받을 수 있나요?,주민등록등본을 발급하여 팩스로 전송하는 업무는 없습니다.


,순위,유사도,고객질문(요청),상담사답변
0,1,0.5503,이전은 어떻게 하죠?,전국 자동차 등록 관청 및 사업소에서 신청하시면 됩니다.
1,2,0.4947,절차는요?,허가를 받은 후 주소지 상관없이 전국 구청 등에서 개명신고를 하면 됩니다.
2,3,0.4947,절차가 간단한가요?,네 간단합니다. 앱 실행 후 불법주정차유형 선택하고 사진 업로드 및 신고위치를 확인...


,순위,유사도,고객질문(요청),상담사답변
0,1,0.9169,지방세 환급금 신청은 어떻게 해야하죠?,인터넷에서 접수를 하셔야 합니다
1,2,0.7388,환급신청하면 되나요?,"네, 관할 지자체에 환급신청 하시면 됩니다."
2,3,0.5770,지방세 환급에 대해 문의 드려도 될까요?,네 말씀하세요


In [110]:
df2 = df1[['고객질문(요청)', '지식베이스']].applymap(normalize)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_6044\157188984.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2 = df1[['고객질문(요청)', '지식베이스']].applymap(normalize)


In [111]:
df2 = df2.loc[df2['고객질문(요청)'] != '']

In [112]:
df2['지식베이스'].value_counts()

지식베이스
            1772
신청,접수        133
어떻게,방법        74
지원,도움         68
문의,질문         57
            ... 
사이트,공공기관       1
대금,대가          1
테니스장,장소        1
점등             1
지역주민,주민        1
Name: count, Length: 7943, dtype: int64

In [113]:
text_col = "고객질문(요청)"
label_col = "지식베이스"

df2 = df2[[text_col, label_col]].dropna()
df2[text_col] = df2[text_col].astype(str).str.strip()
df2[label_col] = df2[label_col].astype(str).str.strip()
df2 = df2[(df2[text_col] != "") & (df2[label_col] != "")]
df2.head(), df2.shape

(               고객질문(요청)    지식베이스
 0   지방세를 내려면 어떻게 해야됩니까?   지방세,세금
 2         은행 어플에서도 됩니까?  어플,공공기관
 6       은행을 직접방문해도 됩니까?  은행,공공기관
 8      은행위치 좀 알 수 있습니까?  은행,공공기관
 12     버스로 가는 방법도 있습니까?  버스,교통수단,
 (16418, 2))

In [114]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# df2[label_col] = le.fit_transform(df2[label_col])

# # 변환 결과 확인
# print("라벨 매핑:", dict(zip(le.classes_, le.transform(le.classes_))))


In [115]:
from konlpy.tag import Komoran
import re

komoran = Komoran()

def clean_korean(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    text = clean_korean(text)
    return komoran.morphs(text)


In [116]:
from sklearn.model_selection import train_test_split

X = df2[text_col].tolist()
y = df2[label_col].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,  random_state=42
)
len(X_train), len(X_test)


(13134, 3284)

In [117]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC, LinearSVR

# 벡터라이저 + 모델
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        tokenizer=tokenize,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(random_state=42))
])
pipeline


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function tok...00290439BFA60>


In [118]:
pipeline.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, f1_score, classification_report

pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"F1-macro: {f1:.4f}")
print("\n분류 리포트:")
print(classification_report(y_test, pred, target_names=le.classes_))


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\multiclass.py:213: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  y_type = type_of_target(y, input_name="y")


Accuracy: 0.3791
F1-macro: 0.1995

분류 리포트:


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_targ

ValueError: Number of classes, 3238, does not match size of target_names, 7942. Try specifying the labels parameter

In [119]:
demo = [
    "비밀번호를 잊어버렸어요 어떻게 해야 하나요?",
    "배송지를 변경하고 싶은데 가능한가요?"
]
demo_pred = pipeline.predict(demo)
# demo_labels = le.inverse_transform(demo_pred)
list(zip(demo, demo_pred))


[('비밀번호를 잊어버렸어요 어떻게 해야 하나요?', np.str_('비밀번호')),
 ('배송지를 변경하고 싶은데 가능한가요?', np.str_('변경'))]